# LeetCode 3095 — Shortest Subarray With OR at Least K I bằng Python

Link đề: https://leetcode.com/problems/shortest-subarray-with-or-at-least-k-i/ · Độ khó: **Easy** · Chủ đề: Array, Bit Manipulation, Sliding Window

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def minimumSubarrayLength(self, nums: List[int], k: int) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **mảng con liên tiếp ngắn nhất** mà **phép OR bit của mọi phần tử trong nó** `>= k`; nếu không có thì trả về `-1`.
2. Nắm **phép OR theo bit** (`|`), vì sao OR **chỉ lớn lên (không giảm) khi thêm phần tử**, và cách "xem" từng **bit** của một số (`>>`, `&`, `1 << b`).
3. Đi từ cách **thử mọi mảng con và tính OR lại** → **mở rộng từ mỗi điểm bắt đầu và cộng dồn OR** → **cửa sổ trượt có đếm bit (vì OR không thể "hoàn tác")** → **nhớ vị trí xuất hiện gần nhất của mỗi bit** → **giữ các giá trị OR kết thúc tại `i`**.
4. Hiểu mấu chốt: **OR đơn điệu** (thêm phần tử không làm OR nhỏ đi), nhưng **không có phép trừ ngược**; muốn bỏ một phần tử khỏi cửa sổ ta phải **đếm số phần tử đang "bật" từng bit**.
5. Tránh các lỗi kinh điển: nhầm `|` với `+` hoặc `or`, nhầm OR với tổng, quên trường hợp `k = 0` (đáp án `1`), và tính độ dài cửa sổ sai (`right - left + 1`).

## 2. Đề bài & ví dụ

Cho một mảng `nums` gồm các số nguyên **không âm** và một số nguyên `k`.

Một mảng con **không rỗng** (dãy phần tử liên tiếp) được gọi là **đặc biệt** nếu **OR bit của mọi phần tử** trong nó **lớn hơn hoặc bằng `k`**.

Trả về **độ dài của mảng con đặc biệt ngắn nhất**, hoặc `-1` nếu không có mảng con đặc biệt nào.

**Ví dụ 1**

- Input: `nums = [1, 2, 3]`, `k = 2`
- Output: `1`
- Giải thích: mảng con `[3]` có OR bằng `3 >= 2`, độ dài `1`.

**Ví dụ 2**

- Input: `nums = [2, 1, 8]`, `k = 10`
- Output: `3`
- Giải thích: mảng con `[2, 1, 8]` có OR bằng `2 | 1 | 8 = 11 >= 10`. Không có mảng con ngắn hơn đạt `>= 10`.

**Ví dụ 3**

- Input: `nums = [1, 2]`, `k = 0`
- Output: `1`
- Giải thích: mảng con `[1]` có OR bằng `1 >= 0` (thực ra mọi mảng con đều thỏa nên độ dài ngắn nhất là `1`).

**Ví dụ 4 (tự dựng)**

- Input: `nums = [1, 2]`, `k = 4`
- Output: `-1`
- Giải thích: OR lớn nhất là `1 | 2 = 3 < 4`.

**Ràng buộc:** `1 <= nums.length <= 50`; `0 <= nums[i] <= 50`; `0 <= k < 64`.

In [1]:
from typing import List
import random

examples = [
    ([1, 2, 3], 2, 1),
    ([2, 1, 8], 10, 3),
    ([1, 2], 0, 1),
    ([1, 2], 4, -1),
]
for nums, k, expected in examples:
    print("nums =", nums, "k =", k, "-> kỳ vọng", expected)

nums = [1, 2, 3] k = 2 -> kỳ vọng 1
nums = [2, 1, 8] k = 10 -> kỳ vọng 3
nums = [1, 2] k = 0 -> kỳ vọng 1
nums = [1, 2] k = 4 -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1. Phép OR theo bit: toán tử `|`

Mỗi số nguyên được lưu dưới dạng **nhị phân** (các bit `0`/`1`). `a | b` (OR bit) cho một số mà **mỗi bit bằng `1` nếu bit đó bằng `1` ở `a` _hoặc_ ở `b`**. Ví dụ `2 = 0b0010`, `1 = 0b0001`, `8 = 0b1000`; `2 | 1 | 8 = 0b1011 = 11`. Lưu ý: `|` khác `+` (cộng): `3 | 3 = 3`, còn `3 + 3 = 6`.

In [2]:
print(2 | 1, 2 | 1 | 8)        # 3 11
print(bin(2), bin(1), bin(8), bin(2 | 1 | 8))
print(3 | 3, 3 + 3)            # 3 6 : OR khác cộng
print(5 | 0, 0 | 0)            # 5 0 : OR với 0 không đổi

3 11
0b10 0b1 0b1000 0b1011
3 6
5 0


### 3.2. OR chỉ lớn lên khi thêm phần tử (tính đơn điệu)

`a | x >= a` với mọi số không âm `x`: OR chỉ có thể **bật thêm** bit (từ `0` lên `1`), không bao giờ tắt bit. Vì vậy, **thêm phần tử vào mảng con không làm OR giảm**; ngược lại **bỏ phần tử có thể làm OR giảm**. Đây là điểm khác với tổng (tổng cũng không giảm khi thêm số không âm, nhưng bỏ đi thì ta trừ được; OR thì không có phép "trừ").

In [3]:
cur = 0
for x in [2, 1, 8, 5]:
    new = cur | x
    print(f"{cur:>2} | {x} = {new:>2}  (không giảm: {new >= cur})")
    cur = new

 0 | 2 =  2  (không giảm: True)
 2 | 1 =  3  (không giảm: True)
 3 | 8 = 11  (không giảm: True)
11 | 5 = 15  (không giảm: True)


### 3.3. Xem từng bit: `(x >> b) & 1` và `1 << b`

- `1 << b` là số chỉ có bit thứ `b` bằng `1` (`1 << 3 = 8`).
- `(x >> b) & 1` cho giá trị bit thứ `b` của `x` (`0` hoặc `1`); `x >> b` dịch `x` sang phải `b` bit.
- `x & (1 << b)` khác `0` cũng cho biết bit `b` của `x` có bật hay không.

Vì `nums[i] <= 50 < 64` và `k < 64`, mọi số trong bài chỉ dùng **6 bit** (bit `0..5`).

In [4]:
x = 11                                    # 0b1011
print([(x >> b) & 1 for b in range(6)])    # [1, 1, 0, 1, 0, 0] (bit 0 là bit thấp nhất)
print([bool(x & (1 << b)) for b in range(6)])
print(1 << 3, bin(1 << 3))                 # 8 0b1000

[1, 1, 0, 1, 0, 0]
[True, True, False, True, False, False]
8 0b1000


### 3.4. Mảng con và độ dài

Mảng con `nums[i..j]` (cả hai đầu **gồm**) có độ dài `j - i + 1`. Có `n * (n + 1) / 2` mảng con không rỗng. Nếu không tìm thấy mảng con thỏa thì ta trả `-1`; dùng một giá trị lớn như `float("inf")` hoặc `n + 1` làm "chưa có đáp án" rồi đổi về `-1` ở cuối.

In [5]:
nums = [2, 1, 8]
n = len(nums)
subs = [(i, j) for i in range(n) for j in range(i, n)]
print(len(subs), n * (n + 1) // 2)         # 6 6
print([(i, j, j - i + 1) for i, j in subs])

6 6
[(0, 0, 1), (0, 1, 2), (0, 2, 3), (1, 1, 1), (1, 2, 2), (2, 2, 1)]


### 3.5. Giá trị đầu `inf` và `min`

Để tìm **ngắn nhất** ta khởi tạo `best = float("inf")` rồi mỗi lần thấy mảng con thỏa thì `best = min(best, độ_dài)`. Cuối cùng nếu `best` vẫn là `inf` ⇒ không có mảng con nào ⇒ trả `-1`.

In [6]:
best = float("inf")
for length in [3, 1, 2]:
    best = min(best, length)
print(best)                                # 1
print(-1 if float("inf") == float("inf") else 0)

1
-1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `nums = [2, 1, 8]`, `k = 10`. Liệt kê mọi mảng con và OR của chúng:

| mảng con | OR | `>= 10`? | độ dài |
|---|---|---|---|
| `[2]` | 2 | không | 1 |
| `[1]` | 1 | không | 1 |
| `[8]` | 8 | không | 1 |
| `[2, 1]` | 3 | không | 2 |
| `[1, 8]` | 9 | không | 2 |
| `[2, 1, 8]` | 11 | **có** | 3 |

Chỉ có mảng con dài `3` thỏa nên đáp án `3`.

Nhận xét: với **điểm bắt đầu cố định** `i`, càng kéo dài sang phải thì OR càng lớn. Khi OR chạm `>= k` lần đầu, kéo dài nữa chỉ làm mảng con **dài hơn** mà không cải thiện ⇒ dừng.

In [7]:
nums, k = [2, 1, 8], 10
n = len(nums)
for i in range(n):
    cur = 0
    for j in range(i, n):
        cur |= nums[j]
        print(f"nums[{i}..{j}] = {nums[i:j + 1]}: OR = {cur:>2}, >= {k}? {cur >= k}")
        if cur >= k:
            break              # kéo dài thêm chỉ làm dài hơn

nums[0..0] = [2]: OR =  2, >= 10? False
nums[0..1] = [2, 1]: OR =  3, >= 10? False
nums[0..2] = [2, 1, 8]: OR = 11, >= 10? True
nums[1..1] = [1]: OR =  1, >= 10? False
nums[1..2] = [1, 8]: OR =  9, >= 10? False
nums[2..2] = [8]: OR =  8, >= 10? False


## 5. Cách 1 — Brute Force / baseline: thử mọi mảng con, tính OR lại từ đầu

**Ý tưởng.** Với mọi cặp `i <= j`: tính OR của `nums[i..j]` bằng một vòng lặp riêng (bắt đầu từ `0`); nếu `>= k` thì cập nhật đáp án với `j - i + 1`. Đúng định nghĩa, dễ viết, nhưng tính lại nhiều lần.

In [8]:
def min_length_bruteforce(nums: List[int], k: int) -> int:
    n = len(nums)
    best = float("inf")
    for i in range(n):
        for j in range(i, n):
            cur = 0
            for t in range(i, j + 1):             # tính OR của nums[i..j] từ đầu
                cur |= nums[t]
            if cur >= k:
                best = min(best, j - i + 1)
    return -1 if best == float("inf") else best


for nums, k, expected in examples:
    print(min_length_bruteforce(nums, k), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
3 (kỳ vọng 3 )
1 (kỳ vọng 1 )
-1 (kỳ vọng -1 )


**Độ phức tạp (Cách 1).** Có `O(n²)` mảng con, mỗi mảng tính OR mất `O(n)` ⇒ **Time `O(n³)`**, **Space `O(1)`**. Với `n <= 50` là khoảng `20 000` bước nên vẫn chạy được, nhưng lãng phí.

**Nút thắt:** OR của `nums[i..j]` chỉ hơn OR của `nums[i..j-1]` đúng một phép `| nums[j]`. Ta đang tính lại từ đầu mỗi lần.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — cộng dồn OR.** Với điểm bắt đầu `i` cố định, giữ `cur` là OR của `nums[i..j]` và cập nhật `cur |= nums[j]` khi `j` tăng: mỗi mảng con tốn `O(1)` thay vì `O(n)`.

**Bước 2 — dừng sớm nhờ tính đơn điệu.** Khi `cur >= k` lần đầu ở `j`, mọi `j' > j` chỉ cho mảng con dài hơn nên bỏ ngay (`break`).

**Bước 3 — cửa sổ trượt.** Có thể làm `O(n)` (nhân hằng số `6` bit) nếu ta **không bắt đầu lại từ `i + 1`** mà giữ nguyên cửa sổ: thêm phần tử bên phải, và khi OR `>= k` thì thử **bỏ phần tử bên trái**. Vấn đề: OR **không có phép trừ**, nên ta theo dõi **mỗi bit đang được bao nhiêu phần tử trong cửa sổ "bật"** (một mảng đếm `cnt[b]`); bỏ phần tử nghĩa là giảm `cnt[b]` ở các bit của nó, và OR của cửa sổ là các bit có `cnt[b] > 0`.

In [9]:
# Mảng đếm bit: OR của cửa sổ = các bit có cnt[b] > 0
window = [2, 1, 8]
cnt = [0] * 6
for x in window:
    for b in range(6):
        if (x >> b) & 1:
            cnt[b] += 1
print("cnt =", cnt)                                      # bit 0: 1 (số 1), bit 1: 1 (số 2), bit 3: 1 (số 8)
print("OR =", sum(1 << b for b in range(6) if cnt[b] > 0))   # 11

# bỏ phần tử đầu (số 2): chỉ giảm cnt ở bit của nó
for b in range(6):
    if (2 >> b) & 1:
        cnt[b] -= 1
print("sau khi bỏ 2: OR =", sum(1 << b for b in range(6) if cnt[b] > 0))   # 9 = 1 | 8

cnt = [1, 1, 0, 1, 0, 0]
OR = 11
sau khi bỏ 2: OR = 9


## 7. Các cách tiếp theo

### Cách 2 — Mở rộng từ mỗi điểm bắt đầu, cộng dồn OR, dừng sớm

**Giải thích từng đoạn.**

1. `for i in range(n)`: điểm bắt đầu.
2. `cur = 0`; `for j in range(i, n)`: `cur |= nums[j]`.
3. Nếu `cur >= k`: ghi độ dài `j - i + 1` rồi `break` (dài hơn chỉ tệ hơn).
4. Cuối cùng `best` còn `inf` thì trả `-1`.

In [10]:
def min_length_extend(nums: List[int], k: int) -> int:
    n = len(nums)
    best = float("inf")
    for i in range(n):
        cur = 0
        for j in range(i, n):
            cur |= nums[j]                        # cộng dồn OR
            if cur >= k:
                best = min(best, j - i + 1)
                break                             # kéo dài thêm chỉ làm dài hơn
    return -1 if best == float("inf") else best


for nums, k, expected in examples:
    print(min_length_extend(nums, k), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
3 (kỳ vọng 3 )
1 (kỳ vọng 1 )
-1 (kỳ vọng -1 )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionExtend:
    def minimumSubarrayLength(self, nums: List[int], k: int) -> int:
        n = len(nums)
        best = n + 1
        for i in range(n):
            cur = 0
            for j in range(i, n):
                cur |= nums[j]
                if cur >= k:
                    best = min(best, j - i + 1)
                    break
        return best if best <= n else -1


print(SolutionExtend().minimumSubarrayLength([2, 1, 8], 10))      # 3

3


**Độ phức tạp Cách 2:** **Time `O(n²)`**, **Space `O(1)`**. Với `n <= 50` là tối đa `1275` bước: quá đủ cho phiên bản I.

### Cách 3 — Cửa sổ trượt với mảng đếm bit

**Giải thích từng đoạn.**

1. `cnt[b]` = số phần tử trong cửa sổ có bit `b` bật; `left = 0`.
2. Với mỗi `right`: thêm `nums[right]` (tăng `cnt` ở các bit của nó).
3. `while left <= right và OR(cửa sổ) >= k`: ghi `best = min(best, right - left + 1)`; bỏ `nums[left]` (giảm `cnt`), `left += 1`.
4. Điều kiện `left <= right` cần thiết khi `k = 0`: cửa sổ rỗng có OR `0 >= 0` nhưng **mảng con không được rỗng**.

In [12]:
def min_length_window(nums: List[int], k: int) -> int:
    B = 6                                              # nums[i] <= 50 < 64: dùng 6 bit
    cnt = [0] * B

    def window_or() -> int:
        return sum(1 << b for b in range(B) if cnt[b] > 0)

    def add(x: int, delta: int) -> None:
        for b in range(B):
            if (x >> b) & 1:
                cnt[b] += delta

    best = float("inf")
    left = 0
    for right, x in enumerate(nums):
        add(x, +1)                                     # thêm nums[right]
        while left <= right and window_or() >= k:      # cửa sổ đang đạt -> thử thu hẹp
            best = min(best, right - left + 1)
            add(nums[left], -1)                        # bỏ nums[left]
            left += 1
    return -1 if best == float("inf") else best


for nums, k, expected in examples:
    print(min_length_window(nums, k), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
3 (kỳ vọng 3 )
1 (kỳ vọng 1 )
-1 (kỳ vọng -1 )


**Độ phức tạp Cách 3:** mỗi phần tử vào và ra cửa sổ đúng một lần, mỗi lần tốn `O(B)` với `B = 6` bit (và tính OR cũng `O(B)`) ⇒ **Time `O(n · B)`**, **Space `O(B)`**. Đây là cách tối ưu tổng quát (dùng cho bài II với `n` tới `2 * 10^5`).

### Cách 4 — Nhớ vị trí xuất hiện gần nhất của mỗi bit

**Giải thích.** Với mỗi `right`, gọi `last[b]` là chỉ số **gần nhất** (`<= right`) của phần tử có bit `b` bật (`-1` nếu chưa có). OR của `nums[l..right]` chứa bit `b` ⇔ `last[b] >= l`. Ta muốn `l` **lớn nhất** (mảng con ngắn nhất kết thúc tại `right`) mà OR `>= k`. Chỉ cần thử `l` bằng các giá trị `last[b]` (giảm dần): OR của `nums[l..right]` là các bit có `last[b] >= l`. Trường hợp `k = 0` trả `1` ngay.

In [13]:
def min_length_last_bit(nums: List[int], k: int) -> int:
    if k == 0:
        return 1                                        # mọi mảng con một phần tử đều có OR >= 0
    B = 6
    last = [-1] * B
    best = float("inf")
    for right, x in enumerate(nums):
        for b in range(B):
            if (x >> b) & 1:
                last[b] = right                         # bit b xuất hiện gần nhất ở right
        for l in sorted({p for p in last if p >= 0}, reverse=True):     # thử l = từng vị trí, lớn trước
            cur = sum(1 << b for b in range(B) if last[b] >= l)          # OR của nums[l..right]
            if cur >= k:
                best = min(best, right - l + 1)
                break                                   # l lớn nhất đã tìm được cho right này
    return -1 if best == float("inf") else best


for nums, k, expected in examples:
    print(min_length_last_bit(nums, k), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
3 (kỳ vọng 3 )
1 (kỳ vọng 1 )
-1 (kỳ vọng -1 )


**Độ phức tạp Cách 4:** mỗi `right` thử tối đa `B` giá trị `l`, mỗi lần tính OR `O(B)` ⇒ **Time `O(n · B²)`**, **Space `O(B)`**. Cách này **không cần hoàn tác** phép OR; nhưng nhiều hằng số hơn Cách 3.

### Cách 5 — Giữ các giá trị OR của mọi mảng con kết thúc tại `i`

**Giải thích.** Gọi `ends` là dict `{giá trị OR: chỉ số bắt đầu LỚN NHẤT cho giá trị đó}` của các mảng con **kết thúc tại `i`**. Khi qua phần tử mới `x`: mỗi mục `(v, l)` thành `(v | x, l)` (giữ `l` lớn nhất nếu trùng giá trị), và thêm mục `(x, i)` (mảng con chỉ có `x`). Vì mỗi lần OR chỉ **bật thêm bit**, số giá trị OR khác nhau tối đa `B + 1 = 7` nên dict luôn nhỏ. Với mỗi mục `(v, l)` có `v >= k`, cập nhật đáp án `i - l + 1`.

In [14]:
def min_length_or_set(nums: List[int], k: int) -> int:
    best = float("inf")
    ends = {}                                           # {giá trị OR: chỉ số bắt đầu lớn nhất}
    for i, x in enumerate(nums):
        new_ends = {x: i}                               # mảng con chỉ gồm nums[i]
        for v, l in ends.items():
            nv = v | x
            if nv not in new_ends or l > new_ends[nv]:
                new_ends[nv] = l                        # giữ điểm bắt đầu lớn nhất (mảng con ngắn nhất)
        ends = new_ends
        for v, l in ends.items():
            if v >= k:
                best = min(best, i - l + 1)
    return -1 if best == float("inf") else best


for nums, k, expected in examples:
    print(min_length_or_set(nums, k), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
3 (kỳ vọng 3 )
1 (kỳ vọng 1 )
-1 (kỳ vọng -1 )


**Độ phức tạp Cách 5:** dict có tối đa `B + 1` mục, mỗi phần tử xử lý `O(B)` ⇒ **Time `O(n · B)`**, **Space `O(B)`**.

## 8. TRACE từng bước

Ta trace Cách 3 (cửa sổ trượt) trên `nums = [2, 1, 8]`, `k = 10` và trên `nums = [1, 2, 3]`, `k = 2`.

In [15]:
def trace_window(nums: List[int], k: int) -> int:
    B = 6
    cnt = [0] * B
    left = 0
    best = float("inf")
    print("nums =", nums, "k =", k)

    def window_or():
        return sum(1 << b for b in range(B) if cnt[b] > 0)

    def add(x, d):
        for b in range(B):
            if (x >> b) & 1:
                cnt[b] += d

    for right, x in enumerate(nums):
        add(x, +1)
        print(f"right={right} thêm {x}: cnt={cnt}, OR={window_or()}, cửa sổ={nums[left:right + 1]}")
        while left <= right and window_or() >= k:
            best = min(best, right - left + 1)
            print(f"    OR={window_or()} >= {k}: ghi độ dài {right - left + 1}, bỏ {nums[left]}")
            add(nums[left], -1)
            left += 1
            print(f"    sau khi bỏ: left={left}, cnt={cnt}, OR={window_or()}")
    result = -1 if best == float("inf") else best
    print("đáp án =", result)
    return result


assert trace_window([2, 1, 8], 10) == 3
print()
assert trace_window([1, 2, 3], 2) == 1

nums = [2, 1, 8] k = 10
right=0 thêm 2: cnt=[0, 1, 0, 0, 0, 0], OR=2, cửa sổ=[2]
right=1 thêm 1: cnt=[1, 1, 0, 0, 0, 0], OR=3, cửa sổ=[2, 1]
right=2 thêm 8: cnt=[1, 1, 0, 1, 0, 0], OR=11, cửa sổ=[2, 1, 8]
    OR=11 >= 10: ghi độ dài 3, bỏ 2
    sau khi bỏ: left=1, cnt=[1, 0, 0, 1, 0, 0], OR=9
đáp án = 3

nums = [1, 2, 3] k = 2
right=0 thêm 1: cnt=[1, 0, 0, 0, 0, 0], OR=1, cửa sổ=[1]
right=1 thêm 2: cnt=[1, 1, 0, 0, 0, 0], OR=3, cửa sổ=[1, 2]
    OR=3 >= 2: ghi độ dài 2, bỏ 1
    sau khi bỏ: left=1, cnt=[0, 1, 0, 0, 0, 0], OR=2
    OR=2 >= 2: ghi độ dài 1, bỏ 2
    sau khi bỏ: left=2, cnt=[0, 0, 0, 0, 0, 0], OR=0
right=2 thêm 3: cnt=[1, 1, 0, 0, 0, 0], OR=3, cửa sổ=[3]
    OR=3 >= 2: ghi độ dài 1, bỏ 3
    sau khi bỏ: left=3, cnt=[0, 0, 0, 0, 0, 0], OR=0
đáp án = 1


In [16]:
def trace_or_set(nums: List[int], k: int) -> int:
    best = float("inf")
    ends = {}
    for i, x in enumerate(nums):
        new_ends = {x: i}
        for v, l in ends.items():
            nv = v | x
            if nv not in new_ends or l > new_ends[nv]:
                new_ends[nv] = l
        ends = new_ends
        ok = {v: i - l + 1 for v, l in ends.items() if v >= k}
        if ok:
            best = min(best, min(ok.values()))
        print(f"i={i} x={x}: ends={dict(sorted(ends.items()))}, đạt >= {k}: {ok}, best={best}")
    return -1 if best == float("inf") else best


assert trace_or_set([2, 1, 8], 10) == 3

i=0 x=2: ends={2: 0}, đạt >= 10: {}, best=inf
i=1 x=1: ends={1: 1, 3: 0}, đạt >= 10: {}, best=inf
i=2 x=8: ends={8: 2, 9: 1, 11: 0}, đạt >= 10: {11: 3}, best=3


Bảng trace Cách 3 với `[2, 1, 8]`, `k = 10`:

| right | thêm | OR cửa sổ | `>= 10`? | hành động | left | best |
|---|---|---|---|---|---|---|
| 0 | 2 | 2 | không | giữ | 0 | inf |
| 1 | 1 | 3 | không | giữ | 0 | inf |
| 2 | 8 | 11 | **có** | ghi 3, bỏ `2` ⇒ OR = 9 < 10 | 1 | 3 |

Sau bỏ `2`, OR của `[1, 8]` là `9 < 10` nên dừng thu hẹp. Hết mảng, `best = 3`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (mở rộng từ mỗi điểm bắt đầu): đơn giản, đúng, `O(n²)` rất nhỏ với `n <= 50`. (Bài II cùng tên dùng cửa sổ trượt có đếm bit như Cách 3.)

In [17]:
from typing import List

class Solution:
    def minimumSubarrayLength(self, nums: List[int], k: int) -> int:
        n = len(nums)
        best = n + 1                           # n + 1 nghĩa là "chưa tìm được"
        for i in range(n):
            cur = 0                            # OR của nums[i..j]
            for j in range(i, n):
                cur |= nums[j]
                if cur >= k:
                    best = min(best, j - i + 1)
                    break                      # kéo dài thêm chỉ làm mảng con dài hơn
        return best if best <= n else -1


sol = Solution()
for nums, k, expected in examples:
    print(nums, k, "->", sol.minimumSubarrayLength(nums, k))

[1, 2, 3] 2 -> 1
[2, 1, 8] 10 -> 3
[1, 2] 0 -> 1
[1, 2] 4 -> -1


## 10. Vì sao thuật toán đúng

- **Đơn điệu của OR:** với `i` cố định, `OR(nums[i..j])` không giảm khi `j` tăng. Nên `j` đầu tiên có OR `>= k` cho mảng con **ngắn nhất bắt đầu tại `i`**; các `j` sau cho mảng dài hơn nên có thể bỏ.
- **Phủ mọi điểm bắt đầu:** vòng ngoài thử mọi `i`; đáp án là `min` trên mọi `i`, nên không bỏ sót mảng con ngắn nhất.
- **Cộng dồn đúng:** `cur |= nums[j]` bằng OR của `nums[i..j]` vì OR **kết hợp** (`(a | b) | c = a | (b | c)`) và có phần tử đơn vị `0`.
- **Trường hợp `k = 0`:** `cur >= 0` luôn đúng ngay ở `j = i` nên mọi `i` cho độ dài `1`; đáp án `1`. **Trường hợp không có mảng con thỏa:** `best` giữ nguyên "chưa tìm được" ⇒ `-1`.
- **Cửa sổ trượt (Cách 3):** `cnt[b]` đếm chính xác số phần tử có bit `b` trong cửa sổ, nên "OR của cửa sổ = các bit có `cnt > 0`" đúng sau cả thêm và bớt phần tử.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `B = 6` là số bit.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi mảng con, tính OR lại | `O(n³)` | `O(1)` |
| 2. Mở rộng, cộng dồn OR, dừng sớm | `O(n²)` | `O(1)` |
| 3. Cửa sổ trượt, đếm bit | `O(n · B)` | `O(B)` |
| 4. Vị trí gần nhất của mỗi bit | `O(n · B²)` | `O(B)` |
| 5. Các giá trị OR kết thúc tại `i` | `O(n · B)` | `O(B)` |

Với `n <= 50` mọi cách đều dưới mili-giây; khi `n` lớn (bài II) chỉ Cách 3, 4, 5 còn dùng được.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng.

In [18]:
sol = Solution()
impls = {
    "brute": min_length_bruteforce,
    "extend": min_length_extend,
    "window": min_length_window,
    "last_bit": min_length_last_bit,
    "or_set": min_length_or_set,
    "leetcode": sol.minimumSubarrayLength,
}


def check_all(nums, k, expected, label):
    for key, fn in impls.items():
        data = list(nums)
        actual = fn(data, k)
        assert actual == expected, (label, key, nums, k, actual, expected)
        assert data == nums, ("input bị sửa", key)
    print("PASS", label)


for idx, (nums, k, expected) in enumerate(examples, 1):
    check_all(nums, k, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3
PASS ví dụ 4


### 12.1. Các trường hợp biên

In [19]:
check_all([0], 0, 1, "một phần tử 0, k = 0: mảng con [0] có OR 0 >= 0")
check_all([0], 1, -1, "một phần tử 0, k = 1: không đạt")
check_all([5], 5, 1, "OR đúng bằng k")
check_all([5], 6, -1, "OR nhỏ hơn k")
check_all([0, 0, 0], 0, 1, "toàn 0 với k = 0")
check_all([0, 0, 0], 1, -1, "toàn 0 với k > 0")
check_all([1, 2, 4, 8, 16, 32], 63, 6, "cần OR cả sáu phần tử để đạt 63")
check_all([32, 16, 8, 4, 2, 1], 63, 6, "cùng ý, thứ tự ngược")
check_all([1, 1, 1, 1], 2, -1, "OR luôn là 1")
check_all([3, 3, 3], 3, 1, "một phần tử đã đạt")
check_all([1, 2, 1, 2], 3, 2, "[1, 2] hoặc [2, 1] đạt 3")
check_all([50] * 50, 50, 1, "n = 50, mọi phần tử bằng 50")
check_all([0] * 49 + [50], 50, 1, "phần tử cuối đạt")
check_all([1] + [0] * 48 + [2], 3, 50, "hai phần tử ở hai đầu cách xa: cần cả mảng")

PASS một phần tử 0, k = 0: mảng con [0] có OR 0 >= 0
PASS một phần tử 0, k = 1: không đạt
PASS OR đúng bằng k
PASS OR nhỏ hơn k
PASS toàn 0 với k = 0
PASS toàn 0 với k > 0
PASS cần OR cả sáu phần tử để đạt 63
PASS cùng ý, thứ tự ngược
PASS OR luôn là 1
PASS một phần tử đã đạt
PASS [1, 2] hoặc [2, 1] đạt 3
PASS n = 50, mọi phần tử bằng 50
PASS phần tử cuối đạt
PASS hai phần tử ở hai đầu cách xa: cần cả mảng


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) **dùng tổng thay vì OR**; (b) **chỉ xét OR của cả mảng** (cho ra `n` hoặc `-1`). Ta tạo hàm sai rồi tìm input phân biệt.

In [20]:
def wrong_sum(nums, k):
    n = len(nums)
    best = n + 1
    for i in range(n):
        cur = 0
        for j in range(i, n):
            cur += nums[j]                       # SAI: cộng thay vì OR
            if cur >= k:
                best = min(best, j - i + 1)
                break
    return best if best <= n else -1


def wrong_whole_array_only(nums, k):
    cur = 0
    for x in nums:
        cur |= x
    return len(nums) if cur >= k else -1         # SAI: không tìm mảng con ngắn nhất


bad1 = ([1, 1, 1, 1], 2)        # tổng đạt 2 sau 2 phần tử nhưng OR luôn là 1
bad2 = ([1, 2, 3], 2)           # mảng con ngắn nhất là [3] (độ dài 1), không phải cả mảng
print(wrong_sum(*bad1), "!=", min_length_extend(*bad1))
print(wrong_whole_array_only(*bad2), "!=", min_length_extend(*bad2))
assert wrong_sum(*bad1) != min_length_extend(*bad1)
assert wrong_whole_array_only(*bad2) != min_length_extend(*bad2)

2 != -1
3 != 1


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh input đúng ràng buộc (`n <= 50`, `0 <= nums[i] <= 50`, `0 <= k < 64`) với nhiều dạng: nhiều số `0`, ít bit, nhiều bit. Cách 1 (đúng định nghĩa) là chuẩn.

In [21]:
rng = random.Random(3095)
cnt_neg = 0
for trial in range(2500):
    n = rng.choice([1, 2, 3, 5, 10, 25, 50])
    mode = rng.choice(["sparse", "small", "wide", "bits"])
    if mode == "sparse":
        nums = [rng.choice([0, 0, 0, rng.randint(0, 50)]) for _ in range(n)]
    elif mode == "small":
        nums = [rng.randint(0, 3) for _ in range(n)]
    elif mode == "bits":
        nums = [1 << rng.randint(0, 5) if rng.random() < 0.8 else 0 for _ in range(n)]
        nums = [v if v <= 50 else 32 for v in nums]
    else:
        nums = [rng.randint(0, 50) for _ in range(n)]
    k = rng.randint(0, 63)
    assert 1 <= len(nums) <= 50 and all(0 <= v <= 50 for v in nums) and 0 <= k < 64
    ref = min_length_bruteforce(nums, k)
    cnt_neg += ref == -1
    for key, fn in impls.items():
        assert fn(list(nums), k) == ref, (key, nums, k)
print("PASS 2500 input ngẫu nhiên khớp định nghĩa; số ca đáp án -1:", cnt_neg)
assert cnt_neg > 100

PASS 2500 input ngẫu nhiên khớp định nghĩa; số ca đáp án -1: 1232


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `+` (hoặc `or`) thay cho `|`

In [22]:
def sai_dung_cong(nums, k):
    n = len(nums)
    best = n + 1
    for i in range(n):
        cur = 0
        for j in range(i, n):
            cur += nums[j]                      # SAI: OR chứ không phải tổng
            if cur >= k:
                best = min(best, j - i + 1)
                break
    return best if best <= n else -1


print("sai:", sai_dung_cong([1, 1, 1, 1], 2), "| đúng:", Solution().minimumSubarrayLength([1, 1, 1, 1], 2))
print("`3 or 5` =", 3 or 5, "| `3 | 5` =", 3 | 5)

sai: 2 | đúng: -1
`3 or 5` = 3 | `3 | 5` = 7


**Vì sao sai:** OR bit **không** cộng các giá trị (lặp lại cùng bit không làm tăng). `or` (chữ) là toán tử logic: `3 or 5` là `3` (số đầu tiên "đúng"), hoàn toàn khác `3 | 5 = 7`. **Cách sửa:** dùng `|` và `|=`.

### Lỗi 2: Quên trường hợp `k = 0` với cửa sổ trượt (cho phép mảng con rỗng)

In [23]:
def sai_cua_so_rong(nums, k):
    B = 6
    cnt = [0] * B
    left = 0
    best = float("inf")
    for right, x in enumerate(nums):
        for b in range(B):
            if (x >> b) & 1:
                cnt[b] += 1
        while sum(1 << b for b in range(B) if cnt[b] > 0) >= k:       # SAI: thiếu left <= right
            length = right - left + 1
            best = min(best, length)
            if length == 0:                    # cửa sổ rỗng cũng bị coi là "thỏa": dừng để khỏi lỗi chỉ số
                return best
            for b in range(B):
                if (nums[left] >> b) & 1:
                    cnt[b] -= 1
            left += 1
    return -1 if best == float("inf") else best


print("sai (k = 0):", sai_cua_so_rong([1, 2], 0), "| đúng:", Solution().minimumSubarrayLength([1, 2], 0))

sai (k = 0): 0 | đúng: 1


**Vì sao sai:** với `k = 0` cửa sổ **rỗng** cũng thỏa `OR = 0 >= 0`, và nếu không chặn bằng `left <= right`, ta ghi độ dài `0` hoặc âm (ở trên ra `0`). Đề yêu cầu mảng con **không rỗng**. **Cách sửa:** thêm `left <= right` vào điều kiện `while`, hoặc xử lý riêng `k == 0` trả `1`.

### Lỗi 3: Bỏ phần tử khỏi cửa sổ bằng cách "trừ" OR

In [24]:
def sai_tru_or(nums, k):
    n = len(nums)
    cur = 0
    left = 0
    best = n + 1
    for right, x in enumerate(nums):
        cur |= x
        while left <= right and cur >= k:
            best = min(best, right - left + 1)
            cur -= nums[left]                  # SAI: OR không có phép trừ
            left += 1
    return best if best <= n else -1


print("sai:", sai_tru_or([1, 1, 2], 2), "| đúng:", Solution().minimumSubarrayLength([1, 1, 2], 2))

sai: 2 | đúng: 1


**Vì sao sai:** với `[1, 1, 2]`, `cur = 1 | 1 | 2 = 3`; bỏ phần tử đầu (`1`) bằng cách trừ cho `3 - 1 = 2`, nhưng OR thật của `[1, 2]` vẫn là `3` (bit 0 vẫn còn nhờ phần tử `1` thứ hai). Số `2` sai kéo theo kết quả sai (`2` thay vì `1`). Về nguyên tắc OR **mất thông tin**: không biết bit đó còn được phần tử nào khác "giữ" hay không. **Cách sửa:** đếm số phần tử bật từng bit (Cách 3), hoặc không dùng cửa sổ (Cách 2).

### Lỗi 4: Nhầm độ dài mảng con (`right - left` thay vì `right - left + 1`)

In [25]:
def sai_do_dai(nums, k):
    n = len(nums)
    best = n + 1
    for i in range(n):
        cur = 0
        for j in range(i, n):
            cur |= nums[j]
            if cur >= k:
                best = min(best, j - i)         # SAI: thiếu + 1
                break
    return best if best <= n else -1


print("sai:", sai_do_dai([2, 1, 8], 10), "| đúng:", Solution().minimumSubarrayLength([2, 1, 8], 10))

sai: 2 | đúng: 3


**Vì sao sai:** mảng con `nums[i..j]` có `j - i + 1` phần tử (cả hai đầu gồm). `j - i` đếm thiếu một (mảng con một phần tử sẽ có độ dài `0`). **Cách sửa:** `j - i + 1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "**OR/AND của mảng con**" ⇒ nhớ **đơn điệu**: OR chỉ tăng khi thêm phần tử; AND chỉ giảm. Tận dụng để dừng sớm.
- Cửa sổ trượt với OR cần **đếm bit** (không hoàn tác được). Ý này lặp lại ở nhiều bài bit-manipulation.
- Số bit nhỏ (`< 64` ⇒ 6 bit) ⇒ hằng số `B` nhỏ, thuật toán `O(n · B)` rất nhanh.
- Mẹo "các giá trị OR kết thúc tại `i`" (Cách 5): số giá trị khác nhau tối đa `B + 1`; áp dụng cho nhiều bài đếm mảng con theo OR/AND/GCD.
- Luôn thử `k = 0`, mảng toàn `0`, và `k` lớn hơn OR toàn mảng (đáp án `-1`).

## 15. Biến thể của bài

1. **Shortest Subarray With OR at Least K II (3097):** `n` tới `2 * 10^5`, `nums[i]` tới `10^9`: cần Cách 3 hoặc 5 với `B = 30`.
2. **Trả về chính mảng con** (hoặc chỉ số đầu/cuối).
3. **Đếm số mảng con có OR `>= k`** (không chỉ ngắn nhất).
4. **Bitwise ORs of Subarrays (898)**: đếm số giá trị OR khác nhau của mọi mảng con (dùng ý Cách 5).
5. **Longest Nice Subarray (2401)**: cửa sổ trượt với bit.

In [26]:
def shortest_subarray_range(nums, k):
    n = len(nums)
    best = None
    for i in range(n):
        cur = 0
        for j in range(i, n):
            cur |= nums[j]
            if cur >= k:
                if best is None or j - i + 1 < best[1] - best[0] + 1:
                    best = (i, j)
                break
    return best


def count_subarrays_or_at_least(nums, k):
    n = len(nums)
    total = 0
    for i in range(n):
        cur = 0
        for j in range(i, n):
            cur |= nums[j]
            total += cur >= k
    return total


print(shortest_subarray_range([2, 1, 8], 10))              # (0, 2)
print(shortest_subarray_range([1, 2, 3], 2))               # (2, 2)
print(count_subarrays_or_at_least([2, 1, 8], 10))          # 1
assert shortest_subarray_range([2, 1, 8], 10) == (0, 2)
assert shortest_subarray_range([1, 2], 4) is None
assert count_subarrays_or_at_least([2, 1, 8], 10) == 1
assert count_subarrays_or_at_least([1, 2, 3], 2) == 5      # [2], [3], [1,2], [2,3], [1,2,3]

(0, 2)
(1, 1)
1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi mảng con, tính lại | định nghĩa | `O(n³)` | `O(1)` | kiểm chứng, `n` nhỏ |
| 2. Mở rộng + dừng sớm | cộng dồn OR, `break` | `O(n²)` | `O(1)` | **nên nộp** bài I |
| 3. Cửa sổ trượt đếm bit | `cnt[b]` cho phép "bỏ" phần tử | `O(n · B)` | `O(B)` | `n` lớn (bài II) |
| 4. Vị trí gần nhất của bit | `last[b]`, thử `l` giảm dần | `O(n · B²)` | `O(B)` | khi muốn tránh hoàn tác |
| 5. Giá trị OR kết thúc tại `i` | dict tối đa `B + 1` mục | `O(n · B)` | `O(B)` | đếm / tìm trên mọi mảng con theo OR |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `5 | 2`, `6 | 6`, `0 | 9`, `(13 >> 2) & 1`, `Solution().minimumSubarrayLength([4, 1], 5)`.

**Bài 2 (tính tay).** Với `nums = [1, 4, 2, 8]`, `k = 7`: liệt kê, với mỗi điểm bắt đầu `i`, OR khi kéo dài `j`, rồi cho đáp án.

**Bài 3 (điền TODO).** Hoàn thành `min_length_todo(nums, k)` theo Cách 2.

In [27]:
def min_length_todo(nums, k):
    n = len(nums)
    best = n + 1
    for i in range(n):
        cur = 0
        # TODO 1: duyệt j từ i tới n - 1
        #   TODO 2: cur |= nums[j]
        #   TODO 3: nếu cur >= k: cập nhật best = min(best, j - i + 1) rồi thoát vòng j
        pass
    # TODO 4: trả về best nếu tìm được, ngược lại -1
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `min_length_window2(nums, k)` theo Cách 3 (mảng đếm bit `cnt[0..5]`, hàm `window_or`). Nhớ điều kiện `left <= right`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_special(nums, k)` đếm số mảng con **đặc biệt** (OR `>= k`) bằng Cách 5 (dict các OR kết thúc tại `i`) — không dùng hai vòng lồng. (b) Viết `distinct_ors(nums)` trả về **số giá trị OR khác nhau** của mọi mảng con (liên quan *Bitwise ORs of Subarrays*).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `7`, `6`, `9`, `1` (`13 = 0b1101`, bit 2 bật), `2` (`4 | 1 = 5`). Bài 3: vòng `for j in range(i, n)` rồi `break` khi đạt.
- **Mức 2 (vừa):** Bài 2: `i = 0`: OR `1, 5, 7` ⇒ đạt ở `j = 2` (độ dài 3); `i = 1`: `4, 6, 14` ⇒ đạt ở `j = 3` (độ dài 3); `i = 2`: `2, 10` ⇒ đạt ở `j = 3` (độ dài 2, `10 >= 7`); `i = 3`: `8` ⇒ đạt (độ dài 1). Đáp án `1`. Bài 4: `add(x, ±1)` rồi `while left <= right and window_or() >= k`.
- **Mức 3 (sâu):** Bài 5(a): mỗi `i` cập nhật `ends` như Cách 5; số mảng con kết thúc tại `i` có OR `>= k` là tổng `(số mảng con ứng với mỗi OR)`; ở dict ta chỉ lưu điểm bắt đầu lớn nhất nên cần lưu thêm **số lượng** (đếm) mỗi giá trị OR. 5(b): dùng tập `ends` dạng `set` các giá trị OR kết thúc tại `i`, hợp vào tập chung.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert 5 | 2 == 7
assert 6 | 6 == 6
assert 0 | 9 == 9
assert (13 >> 2) & 1 == 1
assert Solution().minimumSubarrayLength([4, 1], 5) == 2

# Bài 2
assert Solution().minimumSubarrayLength([1, 4, 2, 8], 7) == 1


# Bài 3
def min_length_todo_ans(nums, k):
    n = len(nums)
    best = n + 1
    for i in range(n):
        cur = 0
        for j in range(i, n):
            cur |= nums[j]
            if cur >= k:
                best = min(best, j - i + 1)
                break
    return best if best <= n else -1


for nums, k, expected in examples:
    assert min_length_todo_ans(nums, k) == expected


# Bài 4
def min_length_window2(nums, k):
    cnt = [0] * 6
    left = 0
    best = float("inf")

    def window_or():
        return sum(1 << b for b in range(6) if cnt[b] > 0)

    for right, x in enumerate(nums):
        for b in range(6):
            if (x >> b) & 1:
                cnt[b] += 1
        while left <= right and window_or() >= k:
            best = min(best, right - left + 1)
            for b in range(6):
                if (nums[left] >> b) & 1:
                    cnt[b] -= 1
            left += 1
    return -1 if best == float("inf") else best


for nums, k, expected in examples:
    assert min_length_window2(nums, k) == expected


# Bài 5(a): đếm số mảng con có OR >= k, dùng dict {OR: số mảng con kết thúc tại i}
def count_special(nums, k):
    total = 0
    ends = {}
    for x in nums:
        new_ends = {x: 1}
        for v, c in ends.items():
            new_ends[v | x] = new_ends.get(v | x, 0) + c
        ends = new_ends
        total += sum(c for v, c in ends.items() if v >= k)
    return total


assert count_special([2, 1, 8], 10) == 1
assert count_special([1, 2, 3], 2) == 5
rng = random.Random(5)
for _ in range(300):
    nums = [rng.randint(0, 50) for _ in range(rng.randint(1, 15))]
    k = rng.randint(0, 63)
    assert count_special(nums, k) == count_subarrays_or_at_least(nums, k)


# Bài 5(b)
def distinct_ors(nums):
    result = set()
    ends = set()
    for x in nums:
        ends = {v | x for v in ends} | {x}
        result |= ends
    return len(result)


def distinct_ors_bruteforce(nums):
    vals = set()
    for i in range(len(nums)):
        cur = 0
        for j in range(i, len(nums)):
            cur |= nums[j]
            vals.add(cur)
    return len(vals)


assert distinct_ors([1, 1, 2]) == 3          # 1, 2, 3
for _ in range(300):
    nums = [rng.randint(0, 50) for _ in range(rng.randint(1, 15))]
    assert distinct_ors(nums) == distinct_ors_bruteforce(nums)
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mảng con ngắn nhất có OR `>= k`: cộng dồn OR từ mỗi điểm bắt đầu)

```python
from typing import List

class Solution:
    def minimumSubarrayLength(self, nums: List[int], k: int) -> int:
        n = len(nums)
        best = n + 1
        for i in range(n):
            cur = 0
            for j in range(i, n):
                cur |= nums[j]
                if cur >= k:
                    best = min(best, j - i + 1)
                    break
        return best if best <= n else -1
```

**Checklist**

- [ ] Dùng `|` / `|=` (OR bit), không dùng `+` hay `or`?
- [ ] Dừng (`break`) ngay khi `cur >= k` vì kéo dài chỉ tệ hơn (OR đơn điệu)?
- [ ] Độ dài là `j - i + 1`; mảng con **không rỗng** (đã xử lý `k = 0`)?
- [ ] Trả `-1` khi không có mảng con nào thỏa?
- [ ] Nếu dùng cửa sổ trượt: không "trừ" OR mà đếm bit?
- [ ] Skeleton đúng `class Solution` / `def minimumSubarrayLength(self, nums: List[int], k: int) -> int`?